## Step 1. Data Organisation

This notebook prepares the image dataset for the classification pipeline.
It assumes `dataset_mortalidad.csv` has been generated by `00_dataset_preparation.ipynb`.

### Step 1.1 - Class Stucture

Images are organised into a folder structure by segmentation type and mortality label
(0 = alive, 1 = deceased). Only images present in all three segmentation types **and**
with a valid label in the CSV are included, ensuring a fair and consistent comparison
across experiments.

> **Output:** `label_clasification/` folder with the following structure:
> ```
> label_clasification/
> ├── experts/
> │   ├── 0/   ← survivors
> │   └── 1/   ← deceased
> ├── sam_bbox/
> │   ├── 0/
> │   └── 1/
> └── sam_mask/
>     ├── 0/
>     └── 1/
> ```

In [10]:
import os
import shutil
import pandas as pd

# ===============================
# Paths
# ===============================
csv_path      = "dataset_mortalidad.csv"
segmented_dir = "Segmented_images/Segmented_images"
folders = {
    "experts":  "1_Experts",
    "sam_bbox": "2_SAM_bboxs",
    "sam_mask": "3_SAM_masks"
}
output_dir = "label_clasification"

# ===============================
# Load CSV
# ===============================
df = pd.read_csv(csv_path)
df = df[["Filename", "mortalidad_30d"]]
label_dict = dict(zip(df["Filename"], df["mortalidad_30d"]))

print(f"Pacientes en CSV: {len(label_dict)}")
print(f"Muertes en CSV: {sum(v == 1 for v in label_dict.values())}")

# ===============================
# Get images in each segmentation
# ===============================
imagenes = {}
for key, folder in folders.items():
    path  = os.path.join(segmented_dir, folder)
    files = set([f for f in os.listdir(path) if f.endswith(".jpg")])
    imagenes[key] = files
    print(f"{key}: {len(files)} images")

# ===============================
# Intersection of images
# ===============================
imagenes_comunes = (
    imagenes["experts"]
    & imagenes["sam_bbox"]
    & imagenes["sam_mask"]
)
print("\nImages present in all segmentations:", len(imagenes_comunes))

# ===============================
# DIAGNÓSTICO
# ===============================
en_carpetas_no_csv = imagenes_comunes - set(label_dict.keys())
en_csv_no_carpetas = set(label_dict.keys()) - imagenes_comunes
print(f"\nEn carpetas pero NO en CSV: {len(en_carpetas_no_csv)}")
print(f"En CSV pero NO en carpetas: {len(en_csv_no_carpetas)}")

# ===============================
# Filter images present in CSV — única fuente de verdad
# ===============================
imagenes_finales = [
    img for img in imagenes_comunes
    if img in label_dict
]

n_muertes = sum(1 for img in imagenes_finales if label_dict[img] == 1)
n_vivos   = sum(1 for img in imagenes_finales if label_dict[img] == 0)

print(f"\nImágenes válidas (en carpetas Y en CSV): {len(imagenes_finales)}")
print(f"  Clase 0 (vivos):   {n_vivos}")
print(f"  Clase 1 (muertos): {n_muertes}")

# ===============================
# Copy images by class
# ===============================
for seg_name, folder in folders.items():
    source = os.path.join(segmented_dir, folder)
    for img in imagenes_finales:
        label  = label_dict[img]
        target = os.path.join(output_dir, seg_name, str(label))
        os.makedirs(target, exist_ok=True)
        src = os.path.join(source, img)
        dst = os.path.join(target, img)
        shutil.copyfile(src, dst)

print("\nCopy finished")

# ===============================
# Check final counts per class
# ===============================
print("\n===== CLASS COUNTS =====")
for seg in folders.keys():
    for label in ["0", "1"]:
        path = os.path.join(output_dir, seg, label)
        n    = len([f for f in os.listdir(path) if f.endswith(".jpg")])
        print(f"{seg} / class {label}: {n} images")


Pacientes en CSV: 1112
Muertes en CSV: 98
experts: 1174 images
sam_bbox: 1174 images
sam_mask: 1173 images

Images present in all segmentations: 1173

En carpetas pero NO en CSV: 62
En CSV pero NO en carpetas: 1

Imágenes válidas (en carpetas Y en CSV): 1111
  Clase 0 (vivos):   1013
  Clase 1 (muertos): 98

Copy finished

===== CLASS COUNTS =====
experts / class 0: 1013 images
experts / class 1: 98 images
sam_bbox / class 0: 1013 images
sam_bbox / class 1: 98 images
sam_mask / class 0: 1013 images
sam_mask / class 1: 98 images


### Step 1.2 - DataFrame Construction

A unified DataFrame (`df_all`) is built containing the file path, segmentation type,
and mortality label for every image. This allows filtering by segmentation type before
running each experiment, while keeping a single consistent data structure.

> **Class distribution:** ~1,013 alive (class 0) vs ~98 deceased (class 1) per
> segmentation type, yielding an approximate **10:1 class imbalance** handled
> in the model training via `pos_weight` and log-odds bias initialisation.

> **Output:** `label_clasification/df_all.csv`

In [11]:
import os
import pandas as pd

# ===============================
# Paths
# ===============================
base_dir = "label_clasification"  

folders = {
    "experts": "experts",
    "sam_bbox": "sam_bbox",
    "sam_mask": "sam_mask"
}

# ===============================
# Prepare DataFrame for all segmentations
# ===============================
all_data = []

for seg_name, seg_folder in folders.items():

    data_dir = os.path.join(base_dir, seg_folder)

    for class_label in ["0", "1"]:

        class_dir = os.path.join(data_dir, class_label)

        if not os.path.exists(class_dir):
            print(f"Carpeta no encontrada: {class_dir}")
            continue

        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):

                all_data.append({
                    "segmentation": seg_name,
                    "filename": os.path.join(seg_folder, class_label, fname),
                    "label": int(class_label)
                })

# Creat DataFrame
df_all = pd.DataFrame(all_data)

# Save df_all
df_all.to_csv("label_clasification/df_all.csv", index=False)
print("df_all saved:", df_all.shape)

print("DataFrame prepared for all segmentations:")
print(df_all.head())

#Revision of the number of images per semgentation and class
print("\n=== Count per segmentation and class ===")
print(df_all.groupby(["segmentation", "label"]).size())

df_all saved: (3333, 3)
DataFrame prepared for all segmentations:
  segmentation                                           filename  label
0      experts  experts/0/1.3.12.2.1107.5.3.56.2693.11.2020032...      0
1      experts  experts/0/1.3.76.6.1.1.5.2.3461.4.439320467240...      0
2      experts  experts/0/1.3.51.0.7.1192784042.36252.27462.39...      0
3      experts  experts/0/1.3.12.2.1107.5.3.56.3575.11.2020041...      0
4      experts  experts/0/1.3.12.2.1107.5.3.56.3575.11.2020040...      0

=== Count per segmentation and class ===
segmentation  label
experts       0        1013
              1          98
sam_bbox      0        1013
              1          98
sam_mask      0        1013
              1          98
dtype: int64
